# Test Utils

## Imports

In [1]:
import math
from typing import Literal

import torch
from torch import Tensor

## Geometry

In [43]:
def cross_product_matrix(k: Tensor) -> Tensor:
    r"""Constructs a skew-symmetric matrix (also known as a cross-product matrix) 
    for a given 3D vector $k = [k1, k2, k3]$. The function returns a 
    3x3 skew-symmetric matrix `M(k)` of the form:

    $$
    M(k) = \begin{bmatrix}
    0 & -k_3 & k_2 \\
    k_3 & 0 & -k_1 \\
    -k_2 & k_1 & 0
    \end{bmatrix}
    $$

    Args:
        k: A tensor of shape `[3]` representing the 3D vector.

    Returns:
        A 3x3 skew-symmetric matrix corresponding to the cross-product operation.

    Example:
        >>> k = torch.tensor([1.0, 2.0, 3.0])
        >>> v = torch.tensor([4.0, 5.0, 6.0])
        >>> m = cross_product_matrix(k)
        >>> cross_product = torch.matmul(m, v)
    """

    m = [
        [0, -k[2], k[1]],
        [k[2], 0, -k[0]],
        [-k[1], k[0], 0],
    ]

    return torch.tensor(m, device=k.device)


def rodrigues_rotation_matrix(axis: Tensor, theta_degrees: float) -> Tensor:
    r"""Computes a 3D rotation matrix using Rodrigues' rotation formula.

    This function rotates a vector in 3D space around a specified axis by a
    given angle. The rotation matrix is computed using the following formula:

    $$
    R = I + \sin(\theta)K + (1 - \cos(\theta))K^2
    $$

    Where:
    - \( I \) is the identity matrix.
    - \( K \) is the skew-symmetric matrix (cross-product matrix) derived from the axis of rotation.
    - \( \theta \) is the rotation angle in radians, converted from degrees.

    Args:
        axis: A 3D vector representing the axis of rotation.
        theta_degrees: The angle of rotation in degrees.

    Returns:
        A 3x3 rotation matrix that rotates a vector around the specified axis by the specified angle.
    """
    axis = axis.detach().clone().float()
    axis = axis / axis.norm()
    K = cross_product_matrix(axis)
    t = torch.tensor([theta_degrees / 180.0 * math.pi], device=axis.device)
    R = torch.eye(3, device=axis.device) + torch.sin(t) * K + (1 - torch.cos(t)) * K.mm(K)
    return R


def cross_product_matrices(k: Tensor) -> Tensor:
    r"""Constructs a batch of skew-symmetric matrices (also known as cross-product matrices) 
    for a batch of 3D vectors.

    For each 3D vector \( k = [k_1, k_2, k_3] \), the corresponding skew-symmetric matrix \( K \) is:

    $$
    K = \begin{bmatrix}
    0 & -k_3 & k_2 \\
    k_3 & 0 & -k_1 \\
    -k_2 & k_1 & 0
    \end{bmatrix}
    $$

    Args:
        k: A tensor of shape `[N, 3]` representing N 3D vectors.

    Returns:
        A tensor of shape [N, 3, 3] where each [3, 3] matrix is the skew-symmetric 
        matrix corresponding to the cross-product operation for each vector.
    """
    K = torch.zeros(k.shape[0], 3, 3, device=k.device)
    K[:, 0, 1] = -k[:, 2]
    K[:, 0, 2] = k[:, 1]
    K[:, 1, 0] = k[:, 2]
    K[:, 1, 2] = -k[:, 0]
    K[:, 2, 0] = -k[:, 1]
    K[:, 2, 1] = k[:, 0]

    return K


def rodrigues_rotation_matrices(axes: Tensor, theta_degrees: Tensor) -> Tensor:
    r"""Computes a batch of 3D rotation matrices using Rodrigues' rotation formula.

    Rodrigues' rotation formula for rotating a vector by an angle \( \theta \) around
    an axis \( k \) is given by:

    $$
    R = I + \sin(\theta)K + (1 - \cos(\theta))K^2
    $$

    Where:
    - \( I \) is the identity matrix.
    - \( K \) is the skew-symmetric matrix (cross-product matrix) of the axis \( k \).
    - \( \theta \) is the angle of rotation in radians.

    Args:
        axes: A tensor of shape $[N, 3]$ representing the axes of rotation.
        theta_degrees: A tensor of shape $[N,]$ representing the angles of rotation in degrees.

    Returns:
        A tensor of shape $[N, 3, 3]$ containing the rotation matrices for each axis-angle pair.
    """
    axes = axes / axes.norm(dim=1, keepdim=True)  # Normalize the axes
    theta_radians = theta_degrees * math.pi / 180.0  # Convert angles to radians

    # Create batch of cross-product matrices for the axes
    K = cross_product_matrices(axes)

    # Rodrigues' formula: R = I + sin(theta) * K + (1 - cos(theta)) * K^2
    eye = torch.eye(3, device=axes.device).unsqueeze(0)  # Identity matrix of shape [1, 3, 3]
    sin_theta = torch.sin(theta_radians).unsqueeze(1).unsqueeze(2)  # [N, 1, 1]
    cos_theta = torch.cos(theta_radians).unsqueeze(1).unsqueeze(2)  # [N, 1, 1]

    # Compute the rotation matrix using tensor operations
    R = eye + sin_theta * K + (1 - cos_theta) * K @ K

    return R


def create_3D_rotations(axis: torch.Tensor, angle: torch.Tensor) -> torch.Tensor:
    t1 = torch.cos(angle)
    t2 = 1 - t1
    t3 = axis[:, 0] * axis[:, 0]
    t6 = t2 * axis[:, 0]
    t7 = t6 * axis[:, 1]
    t8 = torch.sin(angle)
    t9 = t8 * axis[:, 2]
    t11 = t6 * axis[:, 2]
    t12 = t8 * axis[:, 1]
    t15 = axis[:, 1] * axis[:, 1]
    t19 = t2 * axis[:, 1] * axis[:, 2]
    t20 = t8 * axis[:, 0]
    t24 = axis[:, 2] * axis[:, 2]

    R = torch.stack(
        [t1 + t2 * t3, t7 - t9, t11 + t12, t7 + t9, t1 + t2 * t15, t19 - t20, t11 - t12, t19 + t20, t1 + t2 * t24],
        dim=1,
    )

    return R.reshape(-1, 3, 3)

In [63]:
# Test case
axes = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
angles = torch.tensor([90.0, 45.0, 180.0]) * torch.pi / 180  # Convert to radians

torch.manual_seed(42)  # Set a seed for reproducibility
axes = torch.rand(5, 3)  # Generate 5 random 3D axes
angles = torch.rand(5) * 2 * torch.pi  # Generate 5 random angles in radians (0 to 2*pi)

axes = torch.tensor([[0.8823, 0.9150, 0.3829],
                     [0.9593, 0.3904, 0.6009],
                     [0.2566, 0.7936, 0.9408],
                     [0.1332, 0.9346, 0.5936],
                     [0.8694, 0.5677, 0.7411]])
angles = torch.tensor([2.6980, 5.5634, 3.6059, 1.6750, 3.9424])
axes = axes / torch.norm(axes, dim=1, keepdim=True)

print(f"{axes = }")
print(f"{angles = }")


# Compute using unified Rodrigues' formula implementation
r1 = create_3D_rotations(axes, angles)
r2 = rodrigues_rotation_matrices(axes, angles)

print(f"{r1 = }")
print(f"{r2 = }")

print(torch.allclose(r1, r2, atol=1e-4))
print(f"\nMax difference: {torch.max(torch.abs(r1 - r2))}")


axes = tensor([[0.6646, 0.6893, 0.2884],
        [0.8012, 0.3260, 0.5018],
        [0.2041, 0.6312, 0.7483],
        [0.1194, 0.8381, 0.5323],
        [0.6815, 0.4450, 0.5809]])
angles = tensor([2.6980, 5.5634, 3.6059, 1.6750, 3.9424])
r1 = tensor([[[-6.2512e-02,  7.4807e-01,  6.6067e-01],
         [ 9.9565e-01,  9.5975e-04,  9.3121e-02],
         [ 6.9027e-02,  6.6362e-01, -7.4488e-01]],

        [[ 9.1116e-01,  3.9562e-01, -1.1520e-01],
         [-2.6603e-01,  7.7832e-01,  5.6873e-01],
         [ 3.1466e-01, -4.8756e-01,  8.1442e-01]],

        [[-8.1523e-01,  5.7909e-01,  6.6127e-03],
         [-9.1075e-02, -1.3947e-01,  9.8603e-01],
         [ 5.7192e-01,  8.0324e-01,  1.6644e-01]],

        [[-8.8264e-02, -4.1890e-01,  9.0373e-01],
         [ 6.3993e-01,  6.7143e-01,  3.7372e-01],
         [-7.6335e-01,  6.1131e-01,  2.0880e-01]],

        [[ 9.1665e-02,  9.3148e-01,  3.5205e-01],
         [ 9.7342e-02, -3.6023e-01,  9.2777e-01],
         [ 9.9102e-01, -5.0775e-02, -1.2369e-01]]])

In [71]:
import time


def benchmark_rotations(batch_size=1000, num_runs=1000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    axes = torch.rand(batch_size, 3, device=device)
    angles_degrees = torch.rand(batch_size, device=device) * 360
    angles_radians = angles_degrees * math.pi / 180

    # Warm-up run
    rodrigues_rotation_matrices(axes, angles_degrees)
    create_3D_rotations(axes, angles_radians)

    # Benchmark rodrigues_rotation_matrices
    start_time = time.time()
    for _ in range(num_runs):
        rodrigues_rotation_matrices(axes, angles_degrees)
    rodrigues_time = (time.time() - start_time) / num_runs

    # Benchmark create_3D_rotations
    start_time = time.time()
    for _ in range(num_runs):
        create_3D_rotations(axes, angles_radians)
    create_3D_time = (time.time() - start_time) / num_runs

    print(f"Average time for rodrigues_rotation_matrices: {rodrigues_time:.6f} seconds")
    print(f"Average time for create_3D_rotations: {create_3D_time:.6f} seconds")
    print(f"Speed-up factor: {rodrigues_time / create_3D_time:.2f}x")

    if rodrigues_time > create_3D_time:
        print(f"create_3D_rotations is faster by a factor of {rodrigues_time / create_3D_time:.2f}x")
    else:
        print(f"rodrigues_rotation_matrices is faster by a factor of {create_3D_time / rodrigues_time:.2f}x")

# Run the benchmark
benchmark_rotations()

Average time for rodrigues_rotation_matrices: 0.000125 seconds
Average time for create_3D_rotations: 0.000117 seconds
Speed-up factor: 1.07x
create_3D_rotations is faster by a factor of 1.07x


In [51]:
r1

tensor([[[ 0.5782,  1.3721,  1.0356],
         [ 1.7007,  0.6902,  0.2881],
         [ 0.2502,  1.0454, -0.6242]],

        [[ 0.9802,  0.4890, -0.1144],
         [-0.3032,  0.7898,  0.6906],
         [ 0.4004, -0.5742,  0.8415]],

        [[-0.7694,  0.8070,  0.1018],
         [-0.0356,  0.2989,  1.5291],
         [ 0.8126,  1.2993,  0.7823]],

        [[-0.0844, -0.4529,  1.0168],
         [ 0.7278,  0.8603,  0.4800],
         [-0.8423,  0.7449,  0.2850]],

        [[ 0.5859,  1.3692,  0.6853],
         [ 0.3051, -0.1495,  1.3378],
         [ 1.5004,  0.0895,  0.2354]]])

In [52]:
r2

tensor([[[-0.0625,  0.7481,  0.6606],
         [ 0.9957,  0.0010,  0.0931],
         [ 0.0690,  0.6636, -0.7449]],

        [[ 0.9112,  0.3956, -0.1152],
         [-0.2660,  0.7783,  0.5687],
         [ 0.3147, -0.4875,  0.8144]],

        [[-0.8152,  0.5791,  0.0065],
         [-0.0911, -0.1394,  0.9860],
         [ 0.5719,  0.8033,  0.1664]],

        [[-0.0882, -0.4189,  0.9037],
         [ 0.6399,  0.6715,  0.3737],
         [-0.7634,  0.6113,  0.2088]],

        [[ 0.0917,  0.9315,  0.3520],
         [ 0.0974, -0.3602,  0.9278],
         [ 0.9910, -0.0508, -0.1237]]])